<a href="https://colab.research.google.com/github/gangambikaavanti/ai_lab_programs/blob/main/week7_unify_1bm24cs101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re

def is_variable(expr):
    """Returns True if the expression is a variable (lowercase string)."""
    return isinstance(expr, str) and expr.islower() and expr.isalpha()

def is_constant(expr):
    """Returns True if the expression is a constant (uppercase string)."""
    return isinstance(expr, str) and expr.isupper() and len(expr) == 1

def occurs_check(var, expr):
    """Returns True if the variable occurs inside the expression."""
    if var == expr:
        return True
    if isinstance(expr, list):
        return any(occurs_check(var, sub_expr) for sub_expr in expr)
    return False

def apply_subst(subst, expr):
    """Applies a substitution dictionary to an expression."""
    if not subst:
        return expr
    if isinstance(expr, str):
        return subst.get(expr, expr)
    if isinstance(expr, list):
        return [apply_subst(subst, sub_expr) for sub_expr in expr]
    return expr

def compose_subst(s_new, s_old):
    """Composes two substitutions (APPEND(S, SUBST))."""
    if s_new == "FAILURE" or s_old == "FAILURE":
        return "FAILURE"
    res = {k: apply_subst(s_new, v) for k, v in s_old.items()}
    for k, v in s_new.items():
        if k not in res:
            res[k] = v
    return res

def unify(psi1, psi2):
    """Performs the Unification Algorithm on two expressions psi1 and psi2."""
    if is_variable(psi1) or is_constant(psi1) or is_variable(psi2) or is_constant(psi2):
        if psi1 == psi2:
            return {}
        elif is_variable(psi1):
            if occurs_check(psi1, psi2):
                return "FAILURE"
            else:
                return {psi1: psi2}
        elif is_variable(psi2):
            if occurs_check(psi2, psi1):
                return "FAILURE"
            else:
                return {psi2: psi1}
        else:
            return "FAILURE"

    if not psi1 or not psi2 or psi1[0] != psi2[0]:
        return "FAILURE"

    if len(psi1) != len(psi2):
        return "FAILURE"

    subst = {}
    for i in range(1, len(psi1)):
        s = unify(psi1[i], psi2[i])
        if s == "FAILURE":
            return "FAILURE"
        if s != {}:
            psi1 = apply_subst(s, psi1)
            psi2 = apply_subst(s, psi2)
            subst = compose_subst(s, subst)

    return subst

def parse_string(s):
    """Parses a string like 'P(x, A)' or 'f(x)' into a list format like ['P', 'x', 'A']."""
    s = s.replace(" ", "")
    tokens = re.findall(r'[a-zA-Z]+|[\(\),]', s)

    def parse_tokens(token_list):
        if not token_list:
            return None
        token = token_list.pop(0)
        if token_list and token_list[0] == '(':
            token_list.pop(0)  # consume '('
            args = [token]
            while token_list and token_list[0] != ')':
                arg = parse_tokens(token_list)
                if arg:
                    args.append(arg)
                if token_list and token_list[0] == ',':
                    token_list.pop(0)  # consume ','
            if token_list:
                token_list.pop(0)  # consume ')'
            return args
        return token

    try:
        return parse_tokens(tokens)
    except Exception:
        return None

# ==========================================
# Terminal Interface
# ==========================================
if __name__ == "__main__":
    print("=== Interactive Unification Algorithm ===")
    print("Type your expressions (e.g., 'P(x, A)' or 'P(B, y)')")
    print("Type 'exit' to quit.\n")

    while True:
        try:
            str1 = input("Enter Expression 1: ").strip()
            if str1.lower() == 'exit':
                break

            str2 = input("Enter Expression 2: ").strip()
            if str2.lower() == 'exit':
                break

            expr1 = parse_string(str1)
            expr2 = parse_string(str2)

            if expr1 is None or expr2 is None:
                print("❌ Error: Invalid format string. Please try again.\n")
                continue

            print(f"-> Parsed internal structures: {expr1} vs {expr2}")
            result = unify(expr1, expr2)
            print(f"🔮 Result: {result}\n")

        except (KeyboardInterrupt, EOFError):
            print("\nExiting...")
            break


=== Interactive Unification Algorithm ===
Type your expressions (e.g., 'P(x, A)' or 'P(B, y)')
Type 'exit' to quit.

Enter Expression 1: P(x,A)
Enter Expression 2: P(B,y)
-> Parsed internal structures: ['P', 'x', 'A'] vs ['P', 'B', 'y']
🔮 Result: {'x': 'B', 'y': 'A'}

Enter Expression 1: P(f(x),A)
Enter Expression 2: P(B,y)
-> Parsed internal structures: ['P', ['f', 'x'], 'A'] vs ['P', 'B', 'y']
🔮 Result: FAILURE

